# Qwen3-8B MAP Training — Corrected QLoRA Notebook

This notebook fixes the main speed and memory issues in the original version:

- uses **QLoRA / LoRA** instead of full fine-tuning
- uses a **single-GPU** setup instead of `device_map="balanced"`
- adds a **train/validation split** with **MAP@3**
- saves **adapter weights + tokenizer + label encoder**, not the full 8B base model
- defaults to **2 epochs**, matching the referenced silver-medal setup more closely

This version is designed for **Colab / local Jupyter with an H100** and will also work on Kaggle with small path edits.

It also includes a **QUICK_RUN** toggle so you can debug the full pipeline on a small subset before launching the full run.


In [1]:
# Install / upgrade dependencies.
# If you already had transformers imported in this runtime, restart the kernel/runtime after this cell,
# then run the notebook again from the top.

!pip install -q -U "transformers>=4.55.0" "accelerate>=0.34.0" "peft>=0.12.0" "bitsandbytes>=0.43.0" datasets scikit-learn joblib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.8/526.8 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 52.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [2]:
import gc
import json
import inspect
import math
import os
import random
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/input")

if IN_COLAB:
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive"):
            drive.mount("/content/drive")
    except Exception:
        pass

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Transformers version:", transformers.__version__)


Torch version: 2.9.0+cu126
CUDA available: True
CUDA device count: 1
GPU: NVIDIA H100 80GB HBM3
Transformers version: 5.3.0


In [3]:
# =========================
# Configuration
# =========================

BASE_MODEL_NAME = "Qwen/Qwen3-8B"

# Data paths: adjust only if auto-detection fails.
TRAIN_CANDIDATES = [
    "/kaggle/input/datasets/mkhanshovon43/map-ds/DS/train.csv",
    "/kaggle/input/map-charting-student-math-misunderstandings/train.csv",
    "/kaggle/input/map-charting-student-math-misunderstandings-csv/train.csv",
]
TEST_CANDIDATES = [
    "/kaggle/input/datasets/mkhanshovon43/map-ds/DS/test.csv",
    "/kaggle/input/map-charting-student-math-misunderstandings/test.csv",
    "/kaggle/input/map-charting-student-math-misunderstandings-csv/test.csv",
]

def first_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    return None

TRAIN_CSV = first_existing(TRAIN_CANDIDATES)
TEST_CSV = first_existing(TEST_CANDIDATES)

# Where to save adapters, tokenizer, metrics, and label encoder.
if IN_COLAB:
    SAVE_ROOT = "/content/drive/MyDrive/map-charting-student-math-misunderstandings/qwen3-8b-qlora"
elif IN_KAGGLE:
    SAVE_ROOT = "/kaggle/working/qwen3-8b-qlora"
else:
    SAVE_ROOT = "./qwen3-8b-qlora"

OUTPUT_DIR = Path(SAVE_ROOT)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# QUICK RUN SWITCH
# -------------------------
# Turn this on to verify the whole pipeline quickly:
# data loading -> prompt formatting -> tokenization -> QLoRA training -> MAP@3 -> saving
QUICK_RUN = False

# These are only used when QUICK_RUN = True
QUICK_RUN_TRAIN_SAMPLES = 2048
QUICK_RUN_VAL_SAMPLES = 512
QUICK_RUN_EPOCHS = 1
QUICK_RUN_MAX_LENGTH = 192
QUICK_RUN_LOGGING_STEPS = 10

# -------------------------
# Full training defaults
# -------------------------
MAX_LENGTH = 256
VAL_SIZE = 0.10
NUM_EPOCHS = 2
LEARNING_RATE = 2e-4
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 1
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01

# 4-bit QLoRA
USE_4BIT = True
USE_DOUBLE_QUANT = True
QUANT_TYPE = "nf4"
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

# Turn this ON only if you still get OOM.
# It saves memory but usually slows training.
USE_GRADIENT_CHECKPOINTING = False

# LoRA
LORA_R = 64
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Logging / saving
LOGGING_STEPS = 50
SAVE_TOTAL_LIMIT = 1

# -------------------------
# Apply QUICK_RUN overrides
# -------------------------
if QUICK_RUN:
    NUM_EPOCHS = QUICK_RUN_EPOCHS
    MAX_LENGTH = QUICK_RUN_MAX_LENGTH
    LOGGING_STEPS = QUICK_RUN_LOGGING_STEPS
    OUTPUT_DIR = OUTPUT_DIR / "quick_run"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_CSV is not None, "Could not find train.csv. Set TRAIN_CSV manually."
print("TRAIN_CSV =", TRAIN_CSV)
print("TEST_CSV  =", TEST_CSV)
print("OUTPUT_DIR =", OUTPUT_DIR)
print("QUICK_RUN =", QUICK_RUN)
if QUICK_RUN:
    print("QUICK_RUN_TRAIN_SAMPLES =", QUICK_RUN_TRAIN_SAMPLES)
    print("QUICK_RUN_VAL_SAMPLES   =", QUICK_RUN_VAL_SAMPLES)
    print("NUM_EPOCHS =", NUM_EPOCHS)
    print("MAX_LENGTH =", MAX_LENGTH)


TRAIN_CSV = /kaggle/input/datasets/mkhanshovon43/map-ds/DS/train.csv
TEST_CSV  = /kaggle/input/datasets/mkhanshovon43/map-ds/DS/test.csv
OUTPUT_DIR = /kaggle/working/qwen3-8b-qlora
QUICK_RUN = False


In [4]:
# =========================
# Data preparation
# =========================

train = pd.read_csv(TRAIN_CSV)

if TEST_CSV is not None and os.path.exists(TEST_CSV):
    test = pd.read_csv(TEST_CSV)
    print("test shape:", test.shape)
else:
    test = None
    print("test.csv not found; training notebook will continue without it.")

print("raw train shape:", train.shape)

train["Misconception"] = train["Misconception"].fillna("NA")
train["StudentExplanation"] = train["StudentExplanation"].fillna("")
train["QuestionText"] = train["QuestionText"].fillna("")
train["MC_Answer"] = train["MC_Answer"].fillna("")

train["target"] = train["Category"].astype(str) + ":" + train["Misconception"].astype(str)

# Infer per-question correct answer from the most frequent True_* answer.
is_true_family = train["Category"].astype(str).str.startswith("True_")
correct = train.loc[is_true_family].copy()
correct["freq"] = correct.groupby(["QuestionId", "MC_Answer"])["MC_Answer"].transform("count")
correct = correct.sort_values(["QuestionId", "freq"], ascending=[True, False])
correct = correct.drop_duplicates(["QuestionId"])[["QuestionId", "MC_Answer"]]
correct["is_correct"] = 1

train = train.merge(correct, on=["QuestionId", "MC_Answer"], how="left")
train["is_correct"] = train["is_correct"].fillna(0).astype(int)

def format_input(row):
    correct_text = "Yes" if int(row["is_correct"]) == 1 else "No"
    return (
        f"Question: {row['QuestionText']}\n"
        f"Answer: {row['MC_Answer']}\n"
        f"Correct? {correct_text}\n"
        f"Student Explanation: {row['StudentExplanation']}"
    )

train["text"] = train.apply(format_input, axis=1)

label_encoder = LabelEncoder()
train["label"] = label_encoder.fit_transform(train["target"])
num_labels = len(label_encoder.classes_)

id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in id2label.items()}

print("num labels:", num_labels)
print(train[["QuestionId", "Category", "Misconception", "is_correct", "label"]].head())
print("\nExample prompt:\n")
print(train.loc[0, "text"])

test shape: (3, 5)
raw train shape: (36696, 7)
num labels: 65
   QuestionId      Category Misconception  is_correct  label
0       31772  True_Correct            NA           1     37
1       31772  True_Correct            NA           1     37
2       31772  True_Neither            NA           1     64
3       31772  True_Neither            NA           1     64
4       31772  True_Correct            NA           1     37

Example prompt:

Question: What fraction of the shape is not shaded? Give your answer in its simplest form. [Image: A triangle split into 9 equal smaller triangles. 6 of them are shaded.]
Answer: \( \frac{1}{3} \)
Correct? Yes
Student Explanation: 0ne third is equal to tree nineth


In [5]:
# =========================
# Train / validation split
# =========================

stratify_labels = train["label"] if train["label"].value_counts().min() >= 2 else None

train_df, val_df = train_test_split(
    train[["text", "label"]].copy(),
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=stratify_labels,
)

def safe_subsample(df, n_samples, seed=SEED):
    if n_samples is None or n_samples >= len(df):
        return df.reset_index(drop=True)

    label_counts = df["label"].value_counts()
    can_stratify = (
        df["label"].nunique() > 1
        and label_counts.min() >= 2
        and n_samples >= df["label"].nunique()
    )
    stratify_y = df["label"] if can_stratify else None

    try:
        sampled_df, _ = train_test_split(
            df,
            train_size=n_samples,
            random_state=seed,
            stratify=stratify_y,
        )
    except Exception:
        sampled_df = df.sample(n=n_samples, random_state=seed)

    return sampled_df.reset_index(drop=True)

if QUICK_RUN:
    train_df = safe_subsample(train_df, QUICK_RUN_TRAIN_SAMPLES)
    val_df = safe_subsample(val_df, QUICK_RUN_VAL_SAMPLES)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("train rows:", len(train_df))
print("val rows:", len(val_df))

if QUICK_RUN:
    print("Running in QUICK_RUN mode.")
    print("This is for fast pipeline validation, not final scoring.")


train rows: 33026
val rows: 3670


In [6]:
# =========================
# Tokenizer + datasets
# =========================

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_NAME,
    trust_remote_code=True,
)

# For decoder-only models, using EOS as PAD avoids resizing embeddings.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_ds = Dataset.from_pandas(train_df.rename(columns={"label": "labels"}), preserve_index=False)
val_ds = Dataset.from_pandas(val_df.rename(columns={"label": "labels"}), preserve_index=False)

train_ds = train_ds.map(tokenize_function, batched=True, remove_columns=["text"])
val_ds = val_ds.map(tokenize_function, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

print(train_ds)
print(val_ds)
print("pad_token:", tokenizer.pad_token, "| pad_token_id:", tokenizer.pad_token_id)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Map:   0%|          | 0/33026 [00:00<?, ? examples/s]

Map:   0%|          | 0/3670 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 33026
})
Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 3670
})
pad_token: <|endoftext|> | pad_token_id: 151643


In [7]:
# =========================
# Model loading: QLoRA / LoRA
# =========================

assert torch.cuda.is_available(), "This notebook expects a GPU runtime."

quantization_config = None
if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=USE_DOUBLE_QUANT,
        bnb_4bit_quant_type=QUANT_TYPE,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    torch_dtype=COMPUTE_DTYPE,
    quantization_config=quantization_config,
    device_map={"": 0},
    trust_remote_code=True,
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model.config.problem_type = "single_label_classification"

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

def find_target_modules(model):
    candidates = {
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    }
    found = set()
    for name, module in model.named_modules():
        last = name.split(".")[-1]
        if last in candidates:
            found.add(last)
    found = sorted(found)
    if not found:
        raise ValueError("Could not find LoRA target modules in the model.")
    return found

def find_modules_to_save(model):
    candidates = ["score", "classifier"]
    found = []
    for candidate in candidates:
        if any(name.endswith(candidate) for name, _ in model.named_modules()):
            found.append(candidate)
    return found or None

target_modules = find_target_modules(model)
modules_to_save = find_modules_to_save(model)

print("target_modules:", target_modules)
print("modules_to_save:", modules_to_save)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=target_modules,
    modules_to_save=modules_to_save,
    bias="none",
)

model = get_peft_model(model, lora_config)

if USE_GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()

def print_trainable_parameters(model):
    trainable = 0
    total = 0
    for _, param in model.named_parameters():
        total += param.numel()
        if param.requires_grad:
            trainable += param.numel()
    pct = 100 * trainable / total
    print(f"trainable params: {trainable:,}")
    print(f"all params:       {total:,}")
    print(f"trainable %:      {pct:.4f}")

print_trainable_parameters(model)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-8B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


target_modules: ['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj']
modules_to_save: ['score']
trainable params: 174,854,144
all params:       4,270,642,176
trainable %:      4.0943


In [8]:
# =========================
# MAP@3 metric
# =========================

def compute_map3(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]

    probs = torch.softmax(torch.tensor(logits), dim=-1).cpu().numpy()
    labels = np.asarray(labels)
    top3 = np.argsort(-probs, axis=1)[:, :3]

    score = 0.0
    for i, y_true in enumerate(labels):
        if top3[i, 0] == y_true:
            score += 1.0
        elif top3[i, 1] == y_true:
            score += 0.5
        elif top3[i, 2] == y_true:
            score += 1.0 / 3.0

    return {"map@3": score / len(labels)}

In [9]:
# =========================
# Trainer
# =========================

def build_training_arguments():
    sig = inspect.signature(TrainingArguments.__init__).parameters

    base_kwargs = {
        "output_dir": str(OUTPUT_DIR / "checkpoints"),
        "overwrite_output_dir": True,
        "num_train_epochs": NUM_EPOCHS,
        "per_device_train_batch_size": TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "lr_scheduler_type": "cosine",
        "bf16": (COMPUTE_DTYPE == torch.bfloat16),
        "fp16": (COMPUTE_DTYPE == torch.float16),
        "logging_steps": LOGGING_STEPS,
        "load_best_model_at_end": True,
        "metric_for_best_model": "map@3",
        "greater_is_better": True,
        "save_total_limit": SAVE_TOTAL_LIMIT,
        "report_to": "none",
        "remove_unused_columns": False,
        "group_by_length": True,
        "dataloader_pin_memory": True,
        "gradient_checkpointing": USE_GRADIENT_CHECKPOINTING,
        "optim": "paged_adamw_8bit",
        "seed": SEED,
        "disable_tqdm": True,
    }

    # Strategy arg names changed across versions.
    if "evaluation_strategy" in sig:
        base_kwargs["evaluation_strategy"] = "epoch"
    elif "eval_strategy" in sig:
        base_kwargs["eval_strategy"] = "epoch"

    if "save_strategy" in sig:
        base_kwargs["save_strategy"] = "epoch"

    if "logging_strategy" in sig:
        base_kwargs["logging_strategy"] = "steps"

    # Filter only supported kwargs.
    filtered_kwargs = {k: v for k, v in base_kwargs.items() if k in sig}

    print("Using TrainingArguments kwargs:")
    print(sorted(filtered_kwargs.keys()))

    # First try the intended optimizer.
    try:
        return TrainingArguments(**filtered_kwargs)
    except TypeError as e:
        print("First TrainingArguments build failed:", e)
        # Fall back to a more widely supported optimizer if needed.
        fallback_kwargs = dict(filtered_kwargs)
        if "optim" in fallback_kwargs:
            fallback_kwargs["optim"] = "adamw_torch"
            print("Retrying with optim='adamw_torch'")
        return TrainingArguments(**fallback_kwargs)

def strip_problematic_notebook_callbacks(trainer_obj):
    removed = []
    for cb in list(trainer_obj.callback_handler.callbacks):
        cb_name = cb.__class__.__name__
        if cb_name == "NotebookProgressCallback":
            trainer_obj.remove_callback(cb.__class__)
            removed.append(cb_name)

    if removed:
        print("Removed problematic callbacks:", removed)
    else:
        print("No problematic notebook callbacks found.")

def safe_evaluate(trainer_obj, eval_dataset=None):
    try:
        return trainer_obj.evaluate(eval_dataset=eval_dataset)
    except RuntimeError as e:
        if "on_train_begin must be called before on_evaluate" in str(e):
            print("Notebook callback bug detected during evaluate(); retrying without notebook callbacks.")
            strip_problematic_notebook_callbacks(trainer_obj)
            return trainer_obj.evaluate(eval_dataset=eval_dataset)
        raise

training_args = build_training_arguments()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_map3,
)

# In some notebook environments, transformers attaches NotebookProgressCallback,
# which can crash on trainer.evaluate() with:
#   RuntimeError: on_train_begin must be called before on_evaluate
# Remove it up front so training/evaluation is stable across Kaggle/Colab/Jupyter.
strip_problematic_notebook_callbacks(trainer)

trainer


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Using TrainingArguments kwargs:
['bf16', 'dataloader_pin_memory', 'disable_tqdm', 'eval_strategy', 'fp16', 'gradient_accumulation_steps', 'gradient_checkpointing', 'greater_is_better', 'learning_rate', 'load_best_model_at_end', 'logging_steps', 'logging_strategy', 'lr_scheduler_type', 'metric_for_best_model', 'num_train_epochs', 'optim', 'output_dir', 'per_device_eval_batch_size', 'per_device_train_batch_size', 'remove_unused_columns', 'report_to', 'save_strategy', 'save_total_limit', 'seed', 'warmup_ratio', 'weight_decay']
No problematic notebook callbacks found.


In [10]:
# =========================
# Train
# =========================

train_result = trainer.train()
train_metrics = train_result.metrics
eval_metrics = safe_evaluate(trainer)

print("Train metrics:", train_metrics)
print("Eval metrics:", eval_metrics)


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'loss': '4.265', 'grad_norm': '19.02', 'learning_rate': '7.903e-05', 'epoch': '0.02421'}
{'loss': '1.346', 'grad_norm': '13.65', 'learning_rate': '0.0001597', 'epoch': '0.04843'}
{'loss': '1.007', 'grad_norm': '13.79', 'learning_rate': '0.0002', 'epoch': '0.07264'}
{'loss': '0.8617', 'grad_norm': '7.382', 'learning_rate': '0.0001998', 'epoch': '0.09685'}
{'loss': '0.7027', 'grad_norm': '4.111', 'learning_rate': '0.0001995', 'epoch': '0.1211'}
{'loss': '0.7005', 'grad_norm': '2.291', 'learning_rate': '0.0001991', 'epoch': '0.1453'}
{'loss': '0.6523', 'grad_norm': '5.479', 'learning_rate': '0.0001984', 'epoch': '0.1695'}
{'loss': '0.6071', 'grad_norm': '2.445', 'learning_rate': '0.0001977', 'epoch': '0.1937'}
{'loss': '0.6007', 'grad_norm': '5.206', 'learning_rate': '0.0001968', 'epoch': '0.2179'}
{'loss': '0.6187', 'grad_norm': '4.655', 'learning_rate': '0.0001957', 'epoch': '0.2421'}
{'loss': '0.5286', 'grad_norm': '2.833', 'learning_rate': '0.0001945', 'epoch': '0.2663'}
{'loss': '0.

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'loss': '0.2508', 'grad_norm': '0.4316', 'learning_rate': '0.0001022', 'epoch': '1.017'}
{'loss': '0.2356', 'grad_norm': '1.514', 'learning_rate': '9.827e-05', 'epoch': '1.041'}
{'loss': '0.2533', 'grad_norm': '2.883', 'learning_rate': '9.436e-05', 'epoch': '1.065'}
{'loss': '0.2503', 'grad_norm': '3.975', 'learning_rate': '9.045e-05', 'epoch': '1.09'}
{'loss': '0.2638', 'grad_norm': '3.82', 'learning_rate': '8.655e-05', 'epoch': '1.114'}
{'loss': '0.2437', 'grad_norm': '3.822', 'learning_rate': '8.268e-05', 'epoch': '1.138'}
{'loss': '0.2523', 'grad_norm': '5.661', 'learning_rate': '7.883e-05', 'epoch': '1.162'}
{'loss': '0.2502', 'grad_norm': '3.962', 'learning_rate': '7.502e-05', 'epoch': '1.186'}
{'loss': '0.2493', 'grad_norm': '3.629', 'learning_rate': '7.124e-05', 'epoch': '1.211'}
{'loss': '0.2462', 'grad_norm': '2.959', 'learning_rate': '6.751e-05', 'epoch': '1.235'}
{'loss': '0.2748', 'grad_norm': '2.994', 'learning_rate': '6.382e-05', 'epoch': '1.259'}
{'loss': '0.2375', 'gr

In [11]:
# =========================
# Save adapter + tokenizer + label encoder + config
# =========================

trainer.save_model(str(OUTPUT_DIR / "adapter"))
tokenizer.save_pretrained(str(OUTPUT_DIR / "adapter"))
joblib.dump(label_encoder, str(OUTPUT_DIR / "label_encoder.joblib"))

config_to_save = {
    "base_model_name": BASE_MODEL_NAME,
    "max_length": MAX_LENGTH,
    "num_labels": num_labels,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "use_4bit": USE_4BIT,
    "quant_type": QUANT_TYPE,
    "compute_dtype": str(COMPUTE_DTYPE),
    "use_gradient_checkpointing": USE_GRADIENT_CHECKPOINTING,
    "val_size": VAL_SIZE,
    "seed": SEED,
    "train_metrics": train_metrics,
    "eval_metrics": eval_metrics,
}

with open(str(OUTPUT_DIR / "run_config.json"), "w") as f:
    json.dump(config_to_save, f, indent=2)

print("Saved to:", OUTPUT_DIR)
print("Adapter folder:", OUTPUT_DIR / "adapter")
print("Label encoder:", OUTPUT_DIR / "label_encoder.joblib")
print("Config:", OUTPUT_DIR / "run_config.json")

Saved to: /kaggle/working/qwen3-8b-qlora
Adapter folder: /kaggle/working/qwen3-8b-qlora/adapter
Label encoder: /kaggle/working/qwen3-8b-qlora/label_encoder.joblib
Config: /kaggle/working/qwen3-8b-qlora/run_config.json


In [12]:
# =========================
# Quick sanity-check inference with the trained in-memory model
# =========================

model.eval()

sample_text = val_df.loc[0, "text"]
inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    logits = model(**inputs).logits

top3_idx = torch.topk(logits, k=3, dim=-1).indices[0].cpu().numpy()
top3_labels = label_encoder.inverse_transform(top3_idx)

print("Sample text:\n")
print(sample_text)
print("\nTop-3 predicted labels:")
for rank, label in enumerate(top3_labels, start=1):
    print(f"{rank}. {label}")

Sample text:

Question: The probability of an event occurring is \( 0.9 \).

Which of the following most accurately describes the likelihood of the event occurring?
Answer: Likely
Correct? Yes
Student Explanation: i think it's b because if it's certain, you're saying that it will happen 100%;however, the number is 0.9 and it isn't "certain" so therefore it would be likely

Top-3 predicted labels:
1. True_Correct:NA
2. True_Neither:NA
3. True_Misconception:Incomplete


## Why this version should be much faster than your original one

Your original notebook was slow because it loaded Qwen3-8B as a full sequence classifier and trained **all** parameters. This notebook instead:

- loads the base model in **4-bit**
- trains only **LoRA adapters + classification head**
- uses a **single GPU placement**
- avoids `device_map="balanced"`
- avoids adding a new PAD token and resizing embeddings
- saves only the **adapter**, not the whole 8B model

If you still want even faster runs, the easiest knobs are:

1. set `USE_GRADIENT_CHECKPOINTING = False`
2. keep `NUM_EPOCHS = 2`
3. reduce `MAX_LENGTH` to `192`
4. lower `VAL_SIZE` a bit during experimentation
5. only lower `TRAIN_BATCH_SIZE` if you hit OOM